# Orbit Wars Episode Scraper (Imitation Learning)

This module downloads episode metadata and replay JSON files from Kaggle for
Orbit Wars imitation learning workflows.

## Reference

This workflow is inspired by the LuxAI2 episode scraper approach from:
https://www.kaggle.com/code/kuto0633/luxai2-episode-scraper-match-downloader/notebook

## Usage

- Build a local dataset of high-quality or own matches.
- Export replay JSON to transform into `(observation, action)` training rows.
- Collect data for debugging purpose

In [ ]:
import json
import logging
from pathlib import Path

import requests

In [ ]:
logging.basicConfig()
logger = logging.getLogger(__name__)

# There is also a replay service url (https://www.kaggle.com/requests/EpisodeService)
# but that one gave 400 errors.
# I can't find the docs of both.
KAGGLE_API_I_BASE = "https://www.kaggle.com/api/i"

LIST_EPISODES_ENDPOINT = f"{KAGGLE_API_I_BASE}/competitions.EpisodeService/ListEpisodes"
# Replay endpoint can be called without XSRF in many cases.
GET_REPLAY_ENDPOINT = f"{KAGGLE_API_I_BASE}/competitions.EpisodeService/GetEpisodeReplay"

## Endpoints

This workflow uses direct `requests` calls to Kaggle internal endpoints:

- `https://www.kaggle.com/api/i/competitions.EpisodeService/ListEpisodes`
- `https://www.kaggle.com/api/i/competitions.EpisodeService/GetEpisodeReplay`

For many environments, replay/list calls work with only payload args. I can't find the documentation of these url's. 

## Output Artifacts

- `episodes/episodes_submission_<submission_id>.json`
    Episode metadata list.
- `episodes/submission_<submission_id>_replays/episode_<episode_id>.json`
    Replay payloads (if `download_replays=True`).

Return value from `run_download_episodes` includes these key paths and counts.

## Example Result

```python
{
    "competition": "orbit-wars",
    "submission_id": 51799179,
    "episodes_count": 50,
    "metadata_path": "episodes/episodes_submission_51799179.json",
    "replay_dir": "episodes/submission_51799179_replays",
    "replays_downloaded": 50,
}
```

In [ ]:
def build_internal_session() -> requests.Session:
    session = requests.Session()
    # make it clear where requests are originating from
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "orbit-wars-imitation-scraper/1.0",
    }

    session.headers.update(headers)
    return session


def _post_json(session: requests.Session, endpoint: str, payload: dict, timeout: int) -> dict:
    """Post and raise if failed, return json"""
    response = session.post(endpoint, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()


def list_episodes(session: requests.Session, submission_id: int, max_episodes: int) -> list[dict]:
    """Create a list of all episodes, loop over pages"""
    episodes: list[dict] = []

    payload = {"submissionId": submission_id}
    data = _post_json(
        session=session,
        endpoint=LIST_EPISODES_ENDPOINT,
        payload=payload,
        timeout=30,
    )
    episodes = data["episodes"]
    return episodes


def fetch_replay(session: requests.Session, episode_id: int) -> dict:
    """Get the replay information"""
    return _post_json(
        session=session,
        endpoint=GET_REPLAY_ENDPOINT,
        payload={"episodeId": episode_id},
        timeout=60,
    )


def run_download_episodes(
    competition: str = "orbit-wars",
    submission_id: int | None = None,
    max_episodes: int = 200,
    download_replays: bool = False,
    output_dir: str = "episodes",
) -> dict:
    """
    Scrape episode list for imitation learning data collection.

    Uses direct requests to Kaggle EpisodeService endpoints.
    submission_id is required.

    Returns:
        dict with summary paths and counts for notebook consumption.
    """
    if submission_id is None:
        raise RuntimeError(
            "submission_id is required. Example: run_download_episodes(submission_id=51799179, ...)"
        )

    out_root = Path(output_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    # reuse the same session
    session = build_internal_session()

    logger.info("Listing episodes for submission %s", submission_id)
    episodes = list_episodes(session, submission_id, max_episodes)
    logger.info("Fetched %d episode metadata records", len(episodes))

    metadata_path = out_root / f"episodes_submission_{submission_id}.json"
    metadata_path.write_text(json.dumps(episodes, indent=2), encoding="utf-8")
    logger.info("Wrote episode metadata to %s", metadata_path)

    replay_dir = out_root / f"submission_{submission_id}_replays"
    downloaded = 0

    if not download_replays:
        return {
            "submission_id": submission_id,
            "episodes_count": len(episodes),
            "metadata_path": str(metadata_path),
            "replay_dir": str(replay_dir),
            "replays_downloaded": downloaded,
        }

    replay_dir.mkdir(parents=True, exist_ok=True)

    for idx, episode in enumerate(episodes, start=1):
        episode_id = int(episode.get("id"))
        replay_path = replay_dir / f"episode_{episode_id}.json"
        if replay_path.exists():
            continue

        replay_payload = fetch_replay(session, episode_id)
        replay_path.write_text(json.dumps(replay_payload), encoding="utf-8")
        downloaded += 1

        if idx % 10 == 0 or idx == len(episodes):
            logger.info("Downloaded %d/%d replays", idx, len(episodes))

    return {
        "competition": competition,
        "submission_id": submission_id,
        "episodes_count": len(episodes),
        "metadata_path": str(metadata_path),
        "replay_dir": str(replay_dir),
        "replays_downloaded": downloaded,
    }

In [ ]:
result = run_download_episodes(
    competition="orbit-wars",
    submission_id=51799179,  # required
    max_episodes=50,
    download_replays=True,
    output_dir="episodes",
)

result

## Collect scraped episodes
Collect the scraped episodes in a single tar file. 

In [ ]:
!tar -czf episodes.tar.gz episodes